In [ ]:
import pickle
import os
from IPython.core.display import display, HTML
import pandas as pd
import sys
import re
from collections import Counter
import shutil
from transformers import AutoTokenizer
from tqdm import tqdm

from normalize_text import normalize

In [ ]:
table_dir = '../data/'
string_matched_data = pickle.load(open(os.path.join(table_dir, 'new_augmented_train_data_verified.pkl'), 'rb'))

lm_name = 'm3rg-iitd/matscibert'
cache_dir = os.path.join(table_dir, '.cache')
tokenizer = AutoTokenizer.from_pretrained(lm_name, cache_dir=cache_dir, model_max_length=512)

In [ ]:
# print(string_matched_data[1].keys(), '\n')
# print(string_matched_data[1])
# use row_label and col_label and table_nums
# 'prop_table', 'prop_names', 'prop_orient', 'props', 'prop_row_col_indexes' -- keys that will be modified

In [ ]:
def find_num(string):
    #remove tabs or unnecessary spaces
    try:
        string = string.strip()
    except:
        pass
    #e.g. already in int or float form: 12.5 -> 12.5
    try:
        if string[0] == '-':
            return float(string[1:])*(-1)
        else:
            return float(string)
    except:
        pass
    # e.g. 99.1x10-7
    range_regex = re.compile('^\d+\.?\d*\s*x\s*\d+\.?\d*[-|+]?\s*\d+\.?\d*$')
    try:
#         print('alla')
        ranges_ut = range_regex.search(string).group().split('x')
        ranges = [x.strip() for x in ranges_ut]
        #print(f'ranges = {ranges}')
        if '-' in ranges[1]:
            numu = ranges[1].split('-')
#             print(ranges)
#             print(float(numu[0]), float(numu[1]))
            num = float(ranges[0]) * pow(float(numu[0]), (-float(numu[1])))
            formatted_result = "{:.3e}".format(num)
            try:
                # Try to convert using literal_eval
                return float(formatted_result)
            except (ValueError, SyntaxError):
                # If literal_eval fails, return None or handle the error as needed
                return None
        elif '+' in ranges[1]:
            numu = ranges[1].split('+')
            num = float(ranges[0]) * pow(float(numu[0]), (float(numu[1])))
#             print(num)
            return num
        elif ranges[1].startswith('10') and 3<=len(ranges[1])<=4 and ranges[1].isdigit():
            numu0 = 10
            numu1 = ranges[1][2:]
            num = float(ranges[0]) * pow(float(numu0), (float(numu1)))
            return num
        num = float(ranges[0]) * float(ranges[1])
        formatted_result = "{:.3e}".format(num)
        return formatted_result
    except:
        pass
    #e.g. 12.5 - 13.5 -> 12.5
    range_regex = re.compile('^\d+\.?\d*\s*-\s*\d+\.?\d*')
    try:
        ranges = range_regex.search(string).group().split('-')
        num = float(ranges[0])
#         print(num)
        return num
    except:
        pass
#     print('alla')
    #e.g. 12.2 (5.2) -> 12.2
    bracket_regex = re.compile('(\d+\.?\d*)\s*\(\d*.?\d*\)')
    try:
        extracted_value = float(bracket_regex.search(string).group(1))
        return float(extracted_value)
    except:
        pass
    #e.g. 12.3 ± 0.5 -> 12.3
    plusmin_regex = re.compile('^(\d+\.?\d*)(\s*[±+-]+\s*\d+\.?\d*)')
    try:
        extracted_value = float(plusmin_regex.search(string).group(1))
        return extracted_value
    except AttributeError:
        pass
    #e.g. <0.05 -> 0.05  |  >72.0 -> 72.0    | ~12 -> 12
    lessthan_roughly_regex = re.compile('([<]|[~]|[>])=?\s*\d+\.*\d*')
    try:
        extracted_value = lessthan_roughly_regex.search(string).group()
        num_regex = re.compile('\d+\.*\d*')
        extracted_value = num_regex.search(extracted_value).group()
        return float(extracted_value)
    except:
        pass
    # e.g. 0.4:0.6 (ratios)
    if ':' in string:
        split = string.split(":")
        try:
            extracted_value = round(float(split[0])/float(split[1]), 3)
            return extracted_value
        except:
            pass
    # e.g. 220 [29] --> 220.0, where citations given, rejecting ab 220 [29] or 7 220 [29]
    if '[' in string and ']' in string:
        try:
            extracted_value = string[:string.index('[')]
            return float(extracted_value)
        except:
            pass
    # e.g. 723 (first peak or other text) --> 723.0
    if '(' in string and ')' in string:
        try:
            extracted_value = string[:string.index('(')]
            return float(extracted_value)
        except:
            pass
    # e.g. '150K' or '150 degC' --> 150.0 but not '1350degC/2h' or 'njn 150 njn'
    try:
        exact_number_regex = re.compile('^(\d+\.?\d*)\s*[a-zA-Z]+$')
        # Using search to find the first occurrence of the pattern in the string
        match = exact_number_regex.search(string)
#         print('Match')
#         print(match)
        
        # If a match is found, extract the numeric part and convert to float
        if match:
            numeric_part = match.group(1)
            extracted_value = float(numeric_part)
            return extracted_value
    except:
        pass
    return None

In [ ]:
num_pattern = re.compile(r'\d*\.\d+|\d+')

def num_post_process(num):
    return float(num)

# old get_nums
# def get_nums(table):
#     nums = []
#     for r in table:
#         r_comps, r_nums = [], []
#         for cell in r:
#             # cell_nums = re.findall(num_pattern, re.sub(cons_pattern, ' ', cell))
#             cell_nums = re.findall(num_pattern, cell)
#             r_nums.append(list(set(map(num_post_process, cell_nums))))
#         nums.append(r_nums)
#     # print(f'nums {nums}')
#     return nums

# new get_nums:
def get_nums(table):
    nums = []
    for r in table:
        r_nums = []
        for cell in r:
            num = find_num(cell)
            if num != None and num != 0:
                cell_nums = [find_num(cell)]
            else:
                cell_nums = []
            r_nums.append(list(set(map(num_post_process, cell_nums))))
        nums.append(r_nums)
    # print(f'nums {nums}')
    return nums

In [ ]:
prop_ids = [510, 1140, 2010, 2051, 540, 50, 180, 60, 160, 1014, 1015, 3012, 3174, 1116, 1119, 1020, 1011, 70, 1306]
prop_names = ['Density', 'GlassTransitionTg', 'RefractiveIndex', 'AbbeValue', 'YoungsModulus', 'ShearModulus', 'VickersHardness', 'PoissonRatio', 'FractureToughness', 'CrystallizationTemp', 'MeltingTemp', 'ElectricConduct', 'DielectricConst', 'TSofP', 'TAnnealingP', 'ExpansionCoeff', 'LiquidusTemperature', 'BulkModulus', 'ActivationEnergy']

In [ ]:
offenders_double_orientation = []
for index, table in enumerate(string_matched_data):
    
    table_vec = []
    for row in table['act_table']:
        table_vec += row
    table_vec = [normalize(cell) for cell in table_vec]
    tok = tokenizer(table_vec, max_length=50, truncation=True)
    table['input_ids'] = tok['input_ids']
    table['attention_mask'] = tok['attention_mask']
    
    table['prop_table'] = False
    table['prop_names'] = set()
    table['prop_orient'] = ''
    table['props'] = []
    table['prop_row_col_indexes'] = []
    table['table_nums'] = get_nums(table['act_table'])
#     table['anno_type'] = {'string_match'}
    
    row_length, col_length = table['num_rows'], table['num_cols']
    row_label, col_label = table['row_label'], table['col_label']
    row_col_label = row_label + col_label
        
    if any(elem>=4 for elem in col_label) and any(elem>=4 for elem in row_label):
        #print(table)
        offenders_double_orientation.append(table['pii'] + '__' + str(table['t_idx']))
        #sys.exit('There is some error, cannot be both orientation')
    elif any(elem>=4 for elem in col_label):
        table['prop_orient'] = 'col'
    elif any(elem>=4 for elem in row_label):
        table['prop_orient'] = 'row'
#     else:
#         sys.exit('There is some error')
        
    if any(elem>=4 for elem in row_col_label):
        table['prop_table'] = True
#     else:
#         sys.exit('There is some error')
        
            
    for i in range(4, 23):
        table['props'].append([[[] for j in range(table['num_cols'])] for i in range(table['num_rows'])])
        table['prop_row_col_indexes'].append([])
        temp_list = []
        found = ''

        # Check if i is in col_label
        if i in col_label and table['prop_orient'] == 'col':
            table['prop_names'].add(prop_names[i-4])
            col_indexes = [index for index, value in enumerate(col_label) if value == i]
            for col_index in col_indexes:
                table['prop_row_col_indexes'][-1].append(col_index)
                for r in range(1, row_length):
                    if len(table['table_nums'][r][col_index])>0:
                        table['props'][-1][r][col_index].append((prop_ids[i-4], table['table_nums'][r][col_index][0], ''))

        # Check if i is in row_label
        elif i in row_label and table['prop_orient'] == 'row':
            table['prop_names'].add(prop_names[i-4])
            row_indexes = [index for index, value in enumerate(row_label) if value == i]
            for row_index in row_indexes:
                table['prop_row_col_indexes'][-1].append(row_index)
                for c in range(1, col_length):
                    if len(table['table_nums'][row_index][c])>0:
                        table['props'][-1][row_index][c].append((prop_ids[i-4], table['table_nums'][row_index][c][0], ''))
                    
#     if index<5:
#         dimensions = get_dimensions(table['props'])
#         print(row_length, col_length)
#         print(dimensions)
                    
    string_matched_data[index]=table

In [ ]:
print(len(string_matched_data))
print(offenders_double_orientation)

In [ ]:
save_data = '../data/data_in_sep_pkl/'
if os.path.exists(save_data):
    shutil.rmtree(save_data)
    print(f"Directory removed.")
os.mkdir(save_data)
print(f"Directory created.")

for table in string_matched_data:
    pii = table['pii']
    tid = table['t_idx']
    save_dir = os.path.join(save_data, pii+'__'+str(tid))
    if not os.path.exists(save_dir):
        os.mkdir(save_dir)
    pickle.dump(table, open(os.path.join(save_dir, "init.pkl"), 'wb'))

In [ ]:
pickle.dump(string_matched_data, open(os.path.join(table_dir, 'train_data_prop_only_anno_verified.pkl'), 'wb'))